# Notebook 03 — SARIMAX & Feature-Based ML (Parts 4–5)

**Prerequisites:** Notebooks 01 and 02.

- **SARIMAX** with temperature + holiday exogenous variables (conditional forecast)
- **HistGradientBoosting** with recursive multi-step forecasting (no leakage)

In [1]:
!pip install -q pandas numpy matplotlib seaborn statsmodels scikit-learn requests holidays tqdm

from google.colab import drive
drive.mount('/content/drive')

import sys
import json
import matplotlib.pyplot as plt

PROJECT_ROOT = "/content/drive/MyDrive/Assignment"
sys.path.insert(0, f"{PROJECT_ROOT}/colab")
from utils import *

paths = get_paths(PROJECT_ROOT)
feature_df = load_processed_weekly(paths)
train, test = train_test_split_series(feature_df["load_gw"], TEST_WEEKS)

with open(paths["checkpoints"] / "sarima_best_params.json") as f:
    best_params = json.load(f)
ORDER = tuple(best_params["order"])
SEASONAL_ORDER = tuple(best_params["seasonal_order"])

print(f"SARIMA order: {ORDER}, seasonal: {SEASONAL_ORDER}")
print(f"Feature table: {feature_df.shape}")

Mounted at /content/drive
SARIMA order: (4, 0, 6), seasonal: (1, 1, 1, 52)
Feature table: (301, 8)


## Part 4 — SARIMAX with temperature & holidays

> **Note:** Future temperature in the test set uses *observed* values → this is a **conditional/explanatory** forecast, not a true operational forecast.

In [2]:
EXOG_COLS = [
    "temp_mean", "heating_degree", "cooling_degree",
    "holiday_days", "has_holiday",
]

y_train = feature_df["load_gw"].iloc[:-TEST_WEEKS]
y_test = feature_df["load_gw"].iloc[-TEST_WEEKS:]
X_train = feature_df[EXOG_COLS].iloc[:-TEST_WEEKS]
X_test = feature_df[EXOG_COLS].iloc[-TEST_WEEKS:]

# SARIMAX without exog (baseline for ablation)
sarimax_none_fit = fit_sarimax(y_train, ORDER, SEASONAL_ORDER, exog_train=None)
fc_none = forecast_sarimax(sarimax_none_fit, len(y_test), y_test.index)

# SARIMAX with temperature only
TEMP_COLS = ["temp_mean", "heating_degree", "cooling_degree"]
sarimax_temp_fit = fit_sarimax(
    y_train, ORDER, SEASONAL_ORDER,
    exog_train=X_train[TEMP_COLS],
)
fc_temp = forecast_sarimax(
    sarimax_temp_fit, len(y_test), y_test.index,
    exog_test=X_test[TEMP_COLS],
)

# SARIMAX with temperature + holidays (full model)
sarimax_full_fit = fit_sarimax(
    y_train, ORDER, SEASONAL_ORDER,
    exog_train=X_train[EXOG_COLS],
)
fc_full = forecast_sarimax(
    sarimax_full_fit, len(y_test), y_test.index,
    exog_test=X_test[EXOG_COLS],
)

metrics = [
    evaluate_forecast("sarimax_no_exog", y_test, fc_none["mean"], y_train),
    evaluate_forecast("sarimax_temp", y_test, fc_temp["mean"], y_train),
    evaluate_forecast("sarimax_temp_holidays", y_test, fc_full["mean"], y_train),
]
print(pd.DataFrame(metrics).round(3).to_string(index=False))

                model   MAE  RMSE  MASE  Bias
      sarimax_no_exog 3.018 3.729 2.255 2.785
         sarimax_temp 2.802 3.589 2.094 2.580
sarimax_temp_holidays 2.782 3.513 2.079 2.652


In [3]:
# SARIMAX forecast plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(y_train.index, y_train, label="Training", color="steelblue")
ax.plot(y_test.index, y_test, label="Test (actual)", color="black", linewidth=2)
ax.plot(y_test.index, fc_full["mean"], label="SARIMAX (temp + holidays)", color="crimson")
if "ci_95" in fc_full:
    ci = fc_full["ci_95"]
    ax.fill_between(y_test.index, ci.iloc[:, 0], ci.iloc[:, 1], alpha=0.15, color="crimson")
ax.set_title("SARIMAX conditional forecast with temperature & holidays")
ax.set_ylabel("GW")
ax.legend()
plt.tight_layout()
plot_and_save(fig, paths["figures"] / "07_sarimax_forecast.png")
plt.show()

## Part 5 — Feature-based ML (recursive forecast, no leakage)

Lag and rolling features use **only past values**. Each forecast step appends the *prediction* (not the actual) to history.

In [4]:
# Build supervised table on training period only for fitting
exog = feature_df[EXOG_COLS]
supervised = make_supervised_features(feature_df["load_gw"], exog=exog)

train_supervised = supervised.loc[supervised.index <= y_train.index[-1]]
feature_cols = [c for c in supervised.columns if c != "load_gw"]

X_tr = train_supervised[feature_cols]
y_tr = train_supervised["load_gw"]

ml_model = fit_feature_model(X_tr, y_tr)

# Recursive forecast over test period
ml_forecast = recursive_ml_forecast(
    ml_model,
    history=y_train,
    exog_full=exog,
    test_index=y_test.index,
    feature_cols=feature_cols,
)

metrics_ml = evaluate_forecast("feature_model_gbr", y_test, ml_forecast, y_train)
print(pd.DataFrame([metrics_ml]).round(3).to_string(index=False))

Recursive ML forecast:   0%|          | 0/104 [00:00<?, ?it/s]

            model   MAE  RMSE  MASE  Bias
feature_model_gbr 2.219 2.962 1.658 1.638


In [5]:
# Combined comparison plot
bench = pd.read_csv(paths["metrics"] / "02_benchmark_metrics.csv")
sarima_m = pd.read_csv(paths["metrics"] / "03_sarima_metrics.csv")

all_metrics = pd.concat([
    bench,
    pd.DataFrame(metrics),
    pd.DataFrame([metrics_ml]),
], ignore_index=True).drop_duplicates(subset=["model"], keep="last")
all_metrics = all_metrics.sort_values("MASE").reset_index(drop=True)
save_metrics(all_metrics.to_dict("records"), paths["metrics"] / "04_all_weekly_metrics.csv")

print("\n=== All weekly model comparison ===")
print(all_metrics.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(y_train.index, y_train, label="Training", color="steelblue")
ax.plot(y_test.index, y_test, label="Test (actual)", color="black", linewidth=2)
ax.plot(y_test.index, ml_forecast, label="Feature model (GBR)", color="darkorange", linewidth=1.5)

# Seasonal naive from notebook 01
weekly = feature_df["load_gw"]
tr, te = train_test_split_series(weekly, TEST_WEEKS)
sn = seasonal_naive_forecast(tr, len(te), te.index)
ax.plot(te.index, sn, label="Seasonal naive", linestyle="--", color="green")

ax.set_title("Weekly model comparison")
ax.set_ylabel("GW")
ax.legend()
plt.tight_layout()
plot_and_save(fig, paths["figures"] / "08_weekly_model_comparison.png")
plt.show()

# Save all weekly forecasts
fc_all = pd.DataFrame({
    "actual": y_test,
    "sarimax_full": fc_full["mean"],
    "feature_model": ml_forecast,
    "seasonal_naive": sn,
})
fc_all.to_csv(paths["forecasts"] / "weekly_all_forecasts.csv")

free_memory()
print("\n✓ Notebook 03 complete. Proceed to 04_LSTM_Hourly.ipynb")


=== All weekly model comparison ===
                model   MAE  RMSE  MASE   Bias
    feature_model_gbr 2.219 2.962 1.658  1.638
       seasonal_naive 2.319 3.007 1.732  1.732
sarimax_temp_holidays 2.782 3.513 2.079  2.652
         sarimax_temp 2.802 3.589 2.094  2.580
      sarimax_no_exog 3.018 3.729 2.255  2.785
                naive 3.783 4.459 2.827 -0.882
                 mean 3.789 4.397 2.831  0.481
                drift 4.340 5.118 3.243  1.007

✓ Notebook 03 complete. Proceed to 04_LSTM_Hourly.ipynb
